In [1]:
import sys

print(sys.executable)

e:\Projekt_GitHub\PortfolioProjekt\.venv\Scripts\python.exe


Damit ist bestätigt, dass Notebook die virtuelle Python-Umgebung des Projekts verwendet.

In [2]:
from pathlib import Path

import pandas as pd

Geprüft, ob pandas verfügbar ist.

In [3]:
DATA_DIR = Path("../data/cleaned")

DATA_DIR.exists()

True

Pfad in der Variablen DATA_DIR gespeichert und deren Existenz geprüft.

In [4]:
list(DATA_DIR.glob("*.csv"))

[WindowsPath('../data/cleaned/abschluesse_schulart.csv'),
 WindowsPath('../data/cleaned/oberschulen_oeffentlich_frei.csv'),
 WindowsPath('../data/cleaned/schuelerausgabensaetze_oberschule.csv'),
 WindowsPath('../data/cleaned/schulen_schueler_lehrer.csv')]

glob("*.csv") sucht alle Dateien im Ordner data/cleaned, deren Name auf .csv endet. Es werden dabei noch keine Daten eingelesen oder verändert.

In [14]:
df_schulen = pd.read_csv(
    DATA_DIR / "oberschulen_oeffentlich_frei.csv",
    sep=";",
)
print(df_schulen.head().T)
print(df_schulen.shape)

                                 0           1           2           3  \
schuljahr                1992/1993   1993/1994   1994/1995   1995/1996   
traegerschaft           öffentlich  öffentlich  öffentlich  öffentlich   
schulen                        661         660         661         657   
schueler_gesamt             222966      216454      217118      220138   
schueler_maennlich          124441      120889      120315      120186   
schueler_weiblich            98525       95565       96803       99952   
lehrpersonen_gesamt          15338       14954       14985       14622   
lehrpersonen_maennlich      4930.0      4687.0      4717.0      4577.0   
lehrpersonen_weiblich        10408       10267       10268       10045   

                                 4  
schuljahr                1996/1997  
traegerschaft           öffentlich  
schulen                        653  
schueler_gesamt             222004  
schueler_maennlich          119757  
schueler_weiblich           102247  


sep=";" bedeutet: Die einzelnen Spalten der CSV sind durch Semikolons getrennt.
.T macht aus den Spalten Zeilen und umgekehrt
Wichtig: Das sind also keine einzelnen Schulen, sondern aggregierte Werte für eine Trägerform in einem Schuljahr.
Anzahl Zeilen(68),Spalten(9)

In [19]:
print(df_schulen["schuljahr"].unique())
print(df_schulen["traegerschaft"].unique())
df_schulen[["schuljahr", "traegerschaft"]].head(5)

<StringArray>
['1992/1993', '1993/1994', '1994/1995', '1995/1996', '1996/1997', '1997/1998',
 '1998/1999', '1999/2000', '2000/2001', '2001/2002', '2002/2003', '2003/2004',
 '2004/2005', '2005/2006', '2006/2007', '2007/2008', '2008/2009', '2009/2010',
 '2010/2011', '2011/2012', '2012/2013', '2013/2014', '2014/2015', '2015/2016',
 '2016/2017', '2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022',
 '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Length: 34, dtype: str
<StringArray>
['öffentlich', 'frei']
Length: 2, dtype: str


,schuljahr,traegerschaft
0,1992/1993,öffentlich
1,1993/1994,öffentlich
2,1994/1995,öffentlich
3,1995/1996,öffentlich
4,1996/1997,öffentlich


es sind 34 + 34 = 68 Zeilen
Es werden aufgrund der bisherigen Struktur jeweils 34 Beobachtungen für öffentlich und frei erwartet.

In [20]:
df_schulen["traegerschaft"].value_counts()

traegerschaft
öffentlich    34
frei          34
Name: count, dtype: int64

In [21]:
df_schulen.dtypes

schuljahr                     str
traegerschaft                 str
schulen                     int64
schueler_gesamt             int64
schueler_maennlich          int64
schueler_weiblich           int64
lehrpersonen_gesamt         int64
lehrpersonen_maennlich    float64
lehrpersonen_weiblich       int64
dtype: object

In [25]:
print(df_schulen.isna().sum())
df_schulen[df_schulen["lehrpersonen_maennlich"].isna()]

schuljahr                 0
traegerschaft             0
schulen                   0
schueler_gesamt           0
schueler_maennlich        0
schueler_weiblich         0
lehrpersonen_gesamt       0
lehrpersonen_maennlich    1
lehrpersonen_weiblich     0
dtype: int64


,schuljahr,traegerschaft,schulen,schueler_gesamt,schueler_maennlich,schueler_weiblich,lehrpersonen_gesamt,lehrpersonen_maennlich,lehrpersonen_weiblich
34,1992/1993,frei,1,81,57,24,4,NaN,4


Datentyp float ist verdächtig, deshalb auf fehlende Werte hin prüfen
Es gibt genau einmal NaN in Zeile 34.

In [26]:
df_schulen.loc[34]

schuljahr                 1992/1993
traegerschaft                  frei
schulen                           1
schueler_gesamt                  81
schueler_maennlich               57
schueler_weiblich                24
lehrpersonen_gesamt               4
lehrpersonen_maennlich          NaN
lehrpersonen_weiblich             4
Name: 34, dtype: object

Es gibt nur 4x weiblich aber kein männlich --> NaN durch "0" ersetzen

In [31]:
df_schulen.loc[34, "lehrpersonen_maennlich"] = 0
df_schulen["lehrpersonen_maennlich"] = (
    df_schulen["lehrpersonen_maennlich"].astype("int64")
)
print(df_schulen.dtypes)

schuljahr                   str
traegerschaft               str
schulen                   int64
schueler_gesamt           int64
schueler_maennlich        int64
schueler_weiblich         int64
lehrpersonen_gesamt       int64
lehrpersonen_maennlich    int64
lehrpersonen_weiblich     int64
dtype: object


NaN durch "0" ersetzt und Datentyp geändert

In [ ]:
print(df_schulen.duplicated().sum())
df_schulen.duplicated(
    subset=["schuljahr", "traegerschaft"]
).sum()

0


np.int64(0)

Es gibt keine doppelten Zeilen.
Werte der beiden Spalten kommen auch nicht mehrfach vor.

In [38]:
print((
    df_schulen["schueler_gesamt"]
    == df_schulen["schueler_maennlich"] + df_schulen["schueler_weiblich"]
).value_counts())
(
    df_schulen["lehrpersonen_gesamt"]
    == df_schulen["lehrpersonen_maennlich"]
    + df_schulen["lehrpersonen_weiblich"]
).value_counts()

True    68
Name: count, dtype: int64


True    68
Name: count, dtype: int64

Prüfung:
Schüler gesamt=mannlich+weiblich
Lehrpersonen gesamt=maännlich+weiblich

In [39]:
df_schulen.info()

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               68 non-null     str  
 1   traegerschaft           68 non-null     str  
 2   schulen                 68 non-null     int64
 3   schueler_gesamt         68 non-null     int64
 4   schueler_maennlich      68 non-null     int64
 5   schueler_weiblich       68 non-null     int64
 6   lehrpersonen_gesamt     68 non-null     int64
 7   lehrpersonen_maennlich  68 non-null     int64
 8   lehrpersonen_weiblich   68 non-null     int64
dtypes: int64(7), str(2)
memory usage: 4.9 KB
